<a href="https://colab.research.google.com/github/kadiwala1234/Assignment-13-Generative-AI-Essentials/blob/main/Generative_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Assignment 13: Generative AI Essentials**

In [1]:
# Methodology: Simple Text Generation Model
# Install Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [2]:
# Load Dataset
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
text = requests.get(url).text

print(text[:1000])

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her

In [3]:
# Preprocessing
# Create character mapping
chars = sorted(list(set(text)))
char_to_idx = {c:i for i, c in enumerate(chars)}
idx_to_char = {i:c for c, i in char_to_idx.items()}

# Create sequences
seq_length = 100
step = 3

sequences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sequences.append(text[i:i+seq_length])
    next_chars.append(text[i+seq_length])

print("Sequences:", len(sequences))

Sequences: 48199


In [4]:
# Vectorization
X = np.zeros((len(sequences), seq_length, len(chars)), dtype=np.bool_)
y = np.zeros((len(sequences), len(chars)), dtype=np.bool_)

for i, seq in enumerate(sequences):
    for t, char in enumerate(seq):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

In [5]:
# Model (LSTM-based)
model = tf.keras.Sequential([
    layers.LSTM(128, input_shape=(seq_length, len(chars))),
    layers.Dense(len(chars), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │       104,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 76)             │         9,804 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 114,764 (448.30 KB)

 Trainable params: 114,764 (448.30 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Training
history = model.fit(X, y, batch_size=128, epochs=10)

Epoch 1/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 91s 237ms/step - loss: 3.0628
Epoch 2/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 86s 229ms/step - loss: 2.5838
Epoch 3/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 87s 231ms/step - loss: 2.3815
Epoch 4/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 87s 231ms/step - loss: 2.2641
Epoch 5/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 88s 232ms/step - loss: 2.1818
Epoch 6/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 141s 230ms/step - loss: 2.1185
Epoch 7/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 87s 232ms/step - loss: 2.0644
Epoch 8/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 87s 231ms/step - loss: 2.0202
Epoch 9/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 142s 232ms/step - loss: 1.9800
Epoch 10/10
377/377 ━━━━━━━━━━━━━━━━━━━━ 141s 231ms/step - loss: 1.9415


In [7]:
# Text Generation Function
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [8]:
def generate_text(seed, length=300):
    generated = seed
    for _ in range(length):
        x = np.zeros((1, seq_length, len(chars)))
        for t, char in enumerate(seed):
            if char in char_to_idx:
                x[0, t, char_to_idx[char]] = 1

        preds = model.predict(x, verbose=0)[0]
        next_index = sample(preds)
        next_char = idx_to_char[next_index]

        generated += next_char
        seed = seed[1:] + next_char

    return generated

In [9]:
# Example Run
print(generate_text("Alice was beginning to ", 300))

Alice was beginning to mneigal,  n r !tulitc,yilneo;,—moleei!tiuy’,et uio lm_f auoa— o haete.sd
;snruual“eotrsdsg ydet
 flsiio.eea_r oynti‘zis?satt i  b iiissftei?tirs uno,trsoit a i?
e e
eataro.ailead
 o o tnsdoske,auaslss
io—ktson  odisewrxiyi!ito
?eoae,tratlpln n ocg nittsiofi c   sprienoct—t n,ynlnpite d-y atru iteoaa
